# RNN Models
Composed of 3 models (one trained on all data, one on critics, one on users)

## Setup
Ensure that pytorch is installed first, as well as any other dependencies

In [1]:
import torch
print(torch.cuda.is_available())   

False


In [ ]:
import copy
import io
import random
import re
import shutil
import time
import unicodedata
import urllib.request
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Subset, random_split, DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Selected device:", DEVICE)

PAD, SOS, EOS, UNK = 0, 1, 2, 3
MAX_SENTENCE_TOKENS = 9
MAX_PAIRS = 3000
BATCH_SIZE = 64

## Preprocessing

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, csv_path, text_col, score_col, vocab=None):
        # Read and drop invalid rows
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=[text_col, score_col])

        self.reviews = df[text_col].astype(str).tolist()
        self.scores = df[score_col].tolist()
        self.vocab = vocab

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, index):
        tokens = tokenize(self.reviews[index])
        token_ids = [self.vocab.get(token, UNK) for token in tokens]

        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(self.scores[index], dtype=torch.float32),
        )

# Need for padding shorter entries
def collate_reviews(batch):
    reviews, scores = zip(*batch)

    lengths = torch.tensor([len(review) for review in reviews])
    reviews = pad_sequence(reviews, batch_first=True, padding_value=PAD)
    scores = torch.stack(scores)

    return reviews, lengths, scores

def make_loaders(review_dataset, batch_size=BATCH_SIZE):
    train_size = int(0.8 * len(review_dataset))
    val_size = int(0.1 * len(review_dataset))
    test_size = len(review_dataset) - train_size - val_size

    train_set, val_set, test_set = random_split(
        review_dataset,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(SEED),
    )

    return (
        DataLoader(
            train_set,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=collate_reviews
        ),
        DataLoader(
            val_set,
            batch_size=batch_size,
            collate_fn=collate_reviews
        ),
        DataLoader(
            test_set,
            batch_size=batch_size,
            collate_fn=collate_reviews
        ),
    )

## RNN Model

### Cell

In [ ]:
class RNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_linear = nn.Linear(input_size, hidden_size)
        self.hidden_linear = nn.Linear(hidden_size, hidden_size)

    def forward(self, x_t, h_prev):
        input = self.input_linear(x_t)
        hidden = self.hidden_linear(h_prev)
        state = input + hidden
        return torch.tanh(state)

### Encoder

In [ ]:
class RecurrentEncoder(nn.Module):
    def __init__(self, source_vocab_size, embedding_dim,
                 hidden_size, dropout=0.1):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(
            source_vocab_size, embedding_dim, padding_idx=PAD
        )
        self.dropout = nn.Dropout(dropout)
        self.cell = RNNCell(input_size=embedding_dim, hidden_size=hidden_size)

    def forward(self, src, src_length):
        embedded = self.dropout(self.embedding(src))
        batch_size = src.size(0)
        h = embedded.new_zeros(batch_size, self.hidden_size)
        for t in range(src.size(1)):
            proposal = self.cell(embedded[:, t], h)
            mask = (t < src_length).unsqueeze(1)
            h = torch.where(mask, proposal, h)
        return h

### Decoder

In [ ]:
class RecurrentDecoder(nn.Module):
    def __init__(self, target_vocab_size, embedding_dim,
                 hidden_size, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(
            target_vocab_size, embedding_dim, padding_idx=PAD
        )
        self.dropout = nn.Dropout(dropout)
        self.cell = RNNCell(input_size=embedding_dim, hidden_size=hidden_size)
        self.output = nn.Linear(hidden_size, target_vocab_size)

    def forward_step(self, tokens, h):
        embedded = self.dropout(self.embedding(tokens))
        h = self.cell(embedded, h)
        logits = self.output(h)
        return logits, h

### Seq2Seq

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size,
                 embedding_dim=64, hidden_size=64, dropout=0.1):
        super().__init__()
        self.encoder =RecurrentEncoder(
            source_vocab_size, embedding_dim, hidden_size, dropout
        )
        self.decoder = RecurrentDecoder(
            target_vocab_size, embedding_dim, hidden_size, dropout
        )

    def forward(self, src, src_lengths, tgt):
        h = self.encoder(src, src_lengths)  
        logits_by_step = []
        for t in range(tgt.shape[1] - 1):
            logits, h = self.decoder.forward_step(tgt[:, t], h)
            logits_by_step.append(logits)
        return torch.stack(logits_by_step, dim=1)

## Training and Validation

### Training One Epoch

In [ ]:
def train_one_epoch(model, loader, criterion,
                    optimizer, device, clip_value=1.0):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_tokens = 0
    for src, src_lengths, tgt in loader:
        src.to(device)
        src_lengths.to(device)
        tgt.to(device)
        optimizer.zero_grad()
        logits = model(src, src_lengths, tgt)
        targets = tgt[:, 1:]
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1)
        )
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=clip_value)
        optimizer.step()
        mask = targets != PAD
        predictions = logits.argmax(dim=-1)
        valid = int(mask.sum().item())
        total_loss += loss.item() * valid
        total_correct += int(
            ((predictions == targets) & mask).sum().item()
        )
        total_tokens += valid
    return total_loss / total_tokens, total_correct / total_tokens

### Evaluate

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_tokens = 0
    for src, src_lengths, tgt in loader:
        src, src_lengths, tgt = src.to(device), src_lengths.to(device), tgt.to(device)
        logits = model(src, src_lengths, tgt)
        targets = tgt[:, 1:]
        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            targets.reshape(-1)
        )
        mask = (targets != PAD)
        predictions = logits.argmax(dim=-1)
        valid = int(mask.sum().item())
        total_loss += float(loss.item()) * valid
        total_correct += int(
            ((predictions == targets) & mask).sum().item()
        )
        total_tokens += valid
    return total_loss / total_tokens, total_correct / total_tokens

### Fit Model

In [ ]:
def fit_model(model, train_loader, val_loader, epochs=6,
              lr=1e-3, clip_value=1.0, device=DEVICE, verbose=True):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD)
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }
    best_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, val_loader,
                                          optimizer, device, clip_value)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        for key, value in (("train_loss", tr_loss), ("train_acc", tr_loss),
                           ("val_loss", va_loss), ("val_acc", va_acc)):
            history[key].append(value)
        if va_loss < best_loss:
            best_loss = va_loss
            best_state = copy.deepcopy(model.state_dict())
        if verbose:
            print(f"Epoch {epoch:02d}/{epochs} | train loss {tr_loss:.4f} | "
                  f"val loss {va_loss:.4f} | train acc {tr_acc:.1%} | val acc {va_acc:.1%}")
    model.load_state_dict(best_state)
    return history